In [1]:
%pip install -q -U transformers accelerate pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unbabel-comet 2.2.7 requires huggingface-hub<1.0,>=0.19.3, but you have huggingface-hub 2.4.0 which is incompatible.
unbabel-comet 2.2.7 requires numpy<2.0.0,>=1.20.0, but you have numpy 2.5.5 which is incompatible.
unbabel-comet 2.2.7 requires transformers<5.0,>=4.17, but you have transformers 5.23.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

In [3]:
import os
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "Necesita GPU"

In [4]:
carpeta = next(c for c in ["/workspace", "/content", "../../reseñas/splits"]
               if os.path.exists(f"{c}/train.csv"))

In [5]:
# Hy-MT2: carga del modelo
modelo_id = "tencent/Hy-MT2-1.8B"

tokenizer = AutoTokenizer.from_pretrained(modelo_id, trust_remote_code=True)
modelo = AutoModelForCausalLM.from_pretrained(
    modelo_id, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
).eval()

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'mscale_all_dim', 'beta_slow', 'beta_fast', 'mscale', 'alpha'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'mscale_all_dim', 'beta_slow', 'beta_fast', 'mscale', 'alpha'}


Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

In [7]:
# Hy-MT2: traducción
def traducir(texto):
    # prompt recomendado
    prompt = (
        "Translate the following text into English. Note that you should "
        "only output the translated result without any additional explanation:\n\n" + texto

    )
    mensajes = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        mensajes, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(modelo.device)

    with torch.no_grad():
        # parámetros recomendados
        outputs = modelo.generate(
            **inputs, max_new_tokens=2048,
            do_sample=True, temperature=0.7, top_p=0.6, top_k=20, repetition_penalty=1.05,
        )

    # solo la traducción
    nuevos = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(nuevos, skip_special_tokens=True).strip()

In [8]:
# Prueba
traducir("La comida estaba deliciosa, pero la atención fue muy lenta.")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


'The food was delicious, but the service was very slow.'

In [10]:
# Traducir español y portugués de cada split 
for split in ["train", "val", "test"]:
    df = pd.read_csv(f"{carpeta}/{split}.csv")
    df.insert(df.columns.get_loc("text") + 1, "text_en", df["text"])

    filas = df["lang"].isin(["es", "pt"])
    df.loc[filas, "text_en"] = [traducir(str(t)) for t in tqdm(df.loc[filas, "text"], desc=split)]

    df.to_csv(f"{carpeta}/{split}_en.csv", index=False)

train:   0%|          | 0/980 [00:00<?, ?it/s]

val:   0%|          | 0/210 [00:00<?, ?it/s]

test:   0%|          | 0/210 [00:00<?, ?it/s]

In [ ]:
# Revisión
ejemplo = pd.read_csv(f"{carpeta}/train_en.csv")
ejemplo[ejemplo["lang"] != "en"][["lang", "text", "text_en", "original"]].head(5)